In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.voos_gold.dim_companhia AS
SELECT * FROM VALUES
  ('TAM','LATAM Brasil','Nacional'), ('GLO','Gol','Nacional'), ('AZU','Azul','Nacional'),
  ('ACN','Azul Conecta','Nacional'), ('PTB','Voepass','Nacional'), ('TTL','Total Linhas Aéreas','Nacional'),
  ('LAN','LATAM Chile','Internacional'), ('LPE','LATAM Peru','Internacional'),
  ('ARE','LATAM Colombia','Internacional'), ('LAP','LATAM Paraguai','Internacional'),
  ('ARG','Aerolíneas Argentinas','Internacional'), ('JAT','JetSMART','Internacional'),
  ('SKU','Sky Airline','Internacional'), ('FBZ','Flybondi','Internacional'),
  ('AVA','Avianca','Internacional'), ('CMP','Copa Airlines','Internacional'),
  ('AAL','American Airlines','Internacional'), ('UAL','United Airlines','Internacional'),
  ('DAL','Delta','Internacional'), ('TAP','TAP Air Portugal','Internacional'),
  ('AFR','Air France','Internacional'), ('KLM','KLM','Internacional'),
  ('IBE','Iberia','Internacional'), ('DLH','Lufthansa','Internacional'),
  ('BAW','British Airways','Internacional'), ('UAE','Emirates','Internacional'),
  ('QTR','Qatar Airways','Internacional'),
  ('JES','JetSMART Argentina','Internacional'), ('LXP','LATAM Express','Internacional'),
  ('ITY','ITA Airways','Internacional'), ('SWR','Swiss','Internacional'),
  ('GTI','Atlas Air','Carga'), ('CJT','Cargojet','Carga'), ('TPA','Avianca Cargo','Carga')
AS t(prefixo, companhia, tipo)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.voos_gold.fato_posicoes AS
SELECT
  p.coleta_utc, p.coleta_brt, p.data_coleta, HOUR(p.coleta_brt) AS hora,
  p.icao24, p.voo,
    CASE
    WHEN c.companhia IS NOT NULL                 THEN c.companhia
    WHEN p.voo IS NULL                           THEN 'Sem identificação'
    WHEN p.voo RLIKE '^P[PRSTU][A-Z]{3}$'        THEN 'Aviação geral (BR)'
    WHEN p.voo RLIKE '^N[0-9]'                   THEN 'Aviação geral (EUA)'
    ELSE 'Outras'
  END AS companhia,
  CASE
    WHEN c.tipo IS NOT NULL                      THEN c.tipo
    WHEN p.voo RLIKE '^P[PRSTU][A-Z]{3}$' OR p.voo RLIKE '^N[0-9]' THEN 'Aviação geral'
    ELSE 'Outros'
  END AS tipo_companhia,
  p.pais_origem, p.latitude, p.longitude, p.altitude_m,
  p.velocidade_kmh, p.direcao_graus, p.taxa_vertical_ms, p.em_solo,
  CASE
    WHEN p.em_solo                THEN 'Em solo'
    WHEN p.taxa_vertical_ms >  2.5 THEN 'Subindo'
    WHEN p.taxa_vertical_ms < -2.5 THEN 'Descendo'
    ELSE 'Cruzeiro'
  END AS fase_voo
FROM workspace.voos_silver.posicoes p
LEFT JOIN workspace.voos_gold.dim_companhia c
  ON SUBSTR(p.voo, 1, 3) = c.prefixo
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.voos_gold.posicao_atual AS
SELECT * FROM workspace.voos_gold.fato_posicoes
WHERE coleta_utc = (SELECT MAX(coleta_utc) FROM workspace.voos_gold.fato_posicoes)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.voos_gold.resumo_coleta AS
SELECT
  coleta_brt, data_coleta, hora, companhia, tipo_companhia,
  COUNT(*)                                        AS avioes,
  SUM(CASE WHEN NOT em_solo THEN 1 ELSE 0 END)    AS no_ar,
  SUM(CASE WHEN em_solo THEN 1 ELSE 0 END)        AS em_solo,
  ROUND(AVG(CASE WHEN NOT em_solo THEN velocidade_kmh END), 0) AS vel_media_kmh,
  ROUND(AVG(CASE WHEN NOT em_solo THEN altitude_m END), 0)     AS alt_media_m
FROM workspace.voos_gold.fato_posicoes
GROUP BY coleta_brt, data_coleta, hora, companhia, tipo_companhia
""")
print("Gold atualizada")

Gold atualizada


In [0]:
display(spark.sql("""
  SELECT companhia, COUNT(*) AS avioes_agora,
         SUM(CASE WHEN fase_voo = 'Cruzeiro' THEN 1 ELSE 0 END) AS cruzeiro,
         SUM(CASE WHEN fase_voo = 'Subindo'  THEN 1 ELSE 0 END) AS subindo,
         SUM(CASE WHEN fase_voo = 'Descendo' THEN 1 ELSE 0 END) AS descendo
  FROM workspace.voos_gold.posicao_atual
  GROUP BY companhia ORDER BY avioes_agora DESC
"""))

companhia,avioes_agora,cruzeiro,subindo,descendo
LATAM Brasil,41,20,5,12
Azul,35,23,5,6
Gol,33,18,6,8
Aviação geral (BR),11,5,2,3
Outras,8,6,1,1
Aerolíneas Argentinas,6,6,0,0
LATAM Chile,6,3,1,2
JetSMART Argentina,5,2,2,1
Sem identificação,3,2,0,0
ITA Airways,2,2,0,0
